### 1 - Kaggle Base

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/bank-marketing.csv", sep=";")

df.head()

#### OVERVIEW

In [ ]:
print("Linhas:", df.shape[0])
print("Colunas:", df.shape[1])

df.info()

In [ ]:
df.describe(include="all").T

#### Missing Values

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
for col in df.select_dtypes(include="object").columns:
    count = (df[col] == "unknown").sum()

    if count > 0:
        print(f"{col}: {count} ({count / len(df) * 100:.2f}%)")

#### Deduplication

In [ ]:
print("Duplicados:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

#### Column drops

In [ ]:
df = df.drop(columns=["duration"])

### Target y

In [ ]:
df["y"].value_counts()

In [ ]:
df["y"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
df["y"].value_counts().plot(kind="bar")

plt.title("Distribuição da conversão")
plt.xlabel("Conversão")
plt.ylabel("Quantidade")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df["converted"] = df["y"].map({
    "yes": 1,
    "no": 0
})

#### Conversion by features

- Job

In [ ]:
conversion_by_job = (
    df.groupby("job")["converted"]
      .mean()
      .sort_values(ascending=False)
)

conversion_by_job

In [ ]:
conversion_by_job.plot(kind="bar", figsize=(10, 5))

plt.title("Taxa de conversão por profissão")
plt.xlabel("Profissão")
plt.ylabel("Taxa de conversão")
plt.xticks(rotation=45)
plt.show()

- Education

In [ ]:
conversion_by_education = (
    df.groupby("education")["converted"]
      .mean()
      .sort_values(ascending=False)
)

conversion_by_education

- Housing

In [ ]:
df.groupby("housing")["converted"].mean()

- loan

In [ ]:
df.groupby("loan")["converted"].mean()

#### Distribution of numeric variables

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

df[numeric_columns].hist(
    figsize=(15, 12),
    bins=20
)

plt.tight_layout()
plt.show()

#### Age and conversion

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=[
        "Até 25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "66+"
    ]
)

In [ ]:
conversion_by_age = (
    df.groupby("age_group", observed=True)["converted"]
      .mean()
)

conversion_by_age

#### Campaign Analysis

In [ ]:
df.groupby("campaign")["converted"].mean().head(15)

In [ ]:
df["campaign"].describe()

### Final Base

In [ ]:
print("Dimensões finais:", df.shape)

df.head()

In [ ]:
df.to_csv(
    "../data/bank-marketing-clean.csv",
    sep=";",
    index=False
)

### 2 - Base prepare

In [ ]:
df["reward"] = df["y"].map({
    "yes": 1,
    "no": 0
})

df = df.drop(columns=["y"])

In [ ]:
df[["reward"]].value_counts()

In [ ]:
ARMS = {
    0: "Oferta A",
    1: "Oferta B",
    2: "Oferta C"
}

### 3 - Baseline

In [ ]:
baseline_conversion = df["reward"].mean()

print(f"Taxa histórica de conversão: {baseline_conversion:.4f}")
print(f"Taxa histórica de conversão: {baseline_conversion * 100:.2f}%")

In [ ]:
def baseline_policy():
    return 0

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["reward"]
)

print("Treino:", train_df.shape)
print("Teste:", test_df.shape)

In [ ]:
features = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed"
]

X = df[features]
y = df["reward"]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

#### Model training

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

model.fit(
    train_df[features],
    train_df["reward"]
)

In [ ]:
probabilities = model.predict_proba(
    test_df[features]
)[:, 1]

probabilities[:10]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report
)

y_pred = (probabilities >= 0.5).astype(int)

y_test = test_df["reward"]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, probabilities))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    probabilities,
    bins=20
)

plt.title("Distribuição das probabilidades de conversão")
plt.xlabel("Probabilidade estimada")
plt.ylabel("Quantidade de clientes")

plt.show()

In [ ]:
results = test_df[features].copy()

results["actual_reward"] = y_test.values
results["predicted_probability"] = probabilities
results["predicted_reward"] = y_pred

results.head(10)

In [ ]:
results.sort_values(
    "predicted_probability",
    ascending=False
).head(10)

In [ ]:
import joblib

joblib.dump(
    model,
    "../src/propensity_model.pkl"
)

In [ ]:
def get_arm_probability(row, base_probability, arm):
    """
    Retorna a probabilidade de conversão simulada
    para uma determinada estratégia.
    """

    probability = base_probability

    # Oferta A: estratégia padrão
    if arm == 0:
        effect = 0.00

    # Oferta B: abordagem personalizada
    elif arm == 1:
        effect = 0.03

        if row["education"] == "tertiary":
            effect += 0.02

        if row["housing"] == "no":
            effect += 0.01

    # Oferta C: abordagem alternativa
    elif arm == 2:
        effect = 0.02

        if row["poutcome"] == "success":
            effect += 0.03

        if row["previous"] > 0:
            effect += 0.01

    else:
        raise ValueError("Braço inválido")

    return np.clip(
        probability + effect,
        0,
        1
    )

In [ ]:
sample = test_df.iloc[0]

sample_probability = probabilities[0]

for arm in ARMS:
    probability = get_arm_probability(
        sample,
        sample_probability,
        arm
    )

    print(
        f"{ARMS[arm]}: {probability:.2%}"
    )

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, probabilities))

In [ ]:
probability_summary = pd.Series(probabilities).describe()

probability_summary

In [ ]:
print(
    f"Probabilidade média estimada: "
    f"{probabilities.mean():.2%}"
)

print(
    f"Probabilidade mínima: "
    f"{probabilities.min():.2%}"
)

print(
    f"Probabilidade máxima: "
    f"{probabilities.max():.2%}"
)

In [ ]:
comparison = pd.DataFrame({
    "actual": y_test.values,
    "probability": probabilities
})

comparison.groupby("actual")["probability"].mean()

In [ ]:
def simulate_reward(row, base_probability, arm):
    """
    Simula o resultado de uma interação com o cliente.

    Retorna:
        1 -> conversão
        0 -> não conversão
    """

    probability = get_arm_probability(
        row,
        base_probability,
        arm
    )

    return np.random.binomial(
        n=1,
        p=probability
    )

In [ ]:
sample = test_df.iloc[0]
sample_probability = probabilities[0]

for arm in ARMS:
    reward = simulate_reward(
        sample,
        sample_probability,
        arm
    )

    print(
        f"{ARMS[arm]} → reward: {reward}"
    )

#### Thompson Sampling

In [ ]:
class ThompsonSampling:
    def __init__(self, n_arms):
        self.n_arms = n_arms

        # α = sucessos + 1
        self.alpha = np.ones(n_arms)

        # β = fracassos + 1
        self.beta = np.ones(n_arms)

    def select_arm(self):
        samples = np.random.beta(
            self.alpha,
            self.beta
        )

        return np.argmax(samples)

    def update(self, arm, reward):
        if reward == 1:
            self.alpha[arm] += 1
        else:
            self.beta[arm] += 1

In [ ]:
bandit = ThompsonSampling(
    n_arms=len(ARMS)
)

for i in range(10):
    arm = bandit.select_arm()

    row = test_df.iloc[i]
    base_probability = probabilities[i]

    reward = simulate_reward(
        row,
        base_probability,
        arm
    )

    bandit.update(
        arm,
        reward
    )

    print(
        f"Cliente {i + 1}: "
        f"{ARMS[arm]} → reward={reward}"
    )

In [ ]:
print("Alpha:", bandit.alpha)
print("Beta:", bandit.beta)

In [ ]:
np.random.seed(42)

In [ ]:
def run_baseline(test_df, probabilities):
    rewards = []

    for i, (_, row) in enumerate(test_df.iterrows()):
        base_probability = probabilities[i]

        reward = simulate_reward(
            row,
            base_probability,
            arm=0
        )

        rewards.append(reward)

    return np.array(rewards)

In [ ]:
baseline_rewards = run_baseline(
    test_df,
    probabilities
)

In [ ]:
baseline_conversion = baseline_rewards.mean()

print(
    f"Baseline - Taxa de conversão: "
    f"{baseline_conversion:.2%}"
)

print(
    f"Baseline - Reward total: "
    f"{baseline_rewards.sum()}"
)

In [ ]:
def run_thompson_sampling(test_df, probabilities):
    bandit = ThompsonSampling(
        n_arms=len(ARMS)
    )

    rewards = []
    selected_arms = []

    for i, (_, row) in enumerate(test_df.iterrows()):

        # Escolhe uma oferta
        arm = bandit.select_arm()

        # Probabilidade base estimada pelo modelo
        base_probability = probabilities[i]

        # Simula a resposta do cliente
        reward = simulate_reward(
            row,
            base_probability,
            arm
        )

        # Atualiza o Thompson Sampling
        bandit.update(
            arm,
            reward
        )

        selected_arms.append(arm)
        rewards.append(reward)

    return (
        np.array(rewards),
        np.array(selected_arms),
        bandit
    )

In [ ]:
ts_rewards, selected_arms, bandit = run_thompson_sampling(
    test_df,
    probabilities
)

In [ ]:
ts_conversion = ts_rewards.mean()

print(
    f"Thompson Sampling - Taxa de conversão: "
    f"{ts_conversion:.2%}"
)

print(
    f"Thompson Sampling - Reward total: "
    f"{ts_rewards.sum()}"
)

In [ ]:
print("Distribuição das ofertas escolhidas:")

for arm, name in ARMS.items():
    count = (selected_arms == arm).sum()

    print(
        f"{name}: {count} "
        f"({count / len(selected_arms):.2%})"
    )

In [ ]:
comparison = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Thompson Sampling"
    ],
    "Conversão": [
        baseline_conversion,
        ts_conversion
    ],
    "Reward": [
        baseline_rewards.sum(),
        ts_rewards.sum()
    ]
})

comparison

In [ ]:
improvement = (
    (ts_conversion - baseline_conversion)
    / baseline_conversion
) * 100

print(
    f"Melhoria relativa: {improvement:.2f}%"
)

In [ ]:
baseline_cumulative = np.cumsum(
    baseline_rewards
)

ts_cumulative = np.cumsum(
    ts_rewards
)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    baseline_cumulative,
    label="Baseline"
)

plt.plot(
    ts_cumulative,
    label="Thompson Sampling"
)

plt.title("Reward acumulado")
plt.xlabel("Interações")
plt.ylabel("Reward acumulado")

plt.legend()
plt.show()

In [ ]:
def calculate_optimal_probabilities(test_df, probabilities):
    optimal_probabilities = []

    for i, (_, row) in enumerate(test_df.iterrows()):
        base_probability = probabilities[i]

        arm_probabilities = [
            get_arm_probability(
                row,
                base_probability,
                arm
            )
            for arm in ARMS
        ]

        optimal_probabilities.append(
            max(arm_probabilities)
        )

    return np.array(optimal_probabilities)

In [ ]:
optimal_probabilities = calculate_optimal_probabilities(
    test_df,
    probabilities
)

In [ ]:
ts_expected_probabilities = np.array([
    get_arm_probability(
        row,
        probabilities[i],
        arm
    )
    for i, (_, row), arm in zip(
        range(len(test_df)),
        test_df.iterrows(),
        selected_arms
    )
])

ts_regret = (
    optimal_probabilities
    - ts_expected_probabilities
)

cumulative_regret = np.cumsum(ts_regret)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    cumulative_regret
)

plt.title("Regret acumulado - Thompson Sampling")
plt.xlabel("Interações")
plt.ylabel("Regret acumulado")

plt.show()

In [ ]:
models = comparison["Modelo"]
conversion = comparison["Conversão"]

plt.figure(figsize=(8, 5))

bars = plt.bar(models, conversion)

plt.title("Comparação da taxa de conversão")
plt.ylabel("Taxa de conversão")
plt.ylim(0, max(conversion) * 1.25)

for bar, value in zip(bars, conversion):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2%}",
        ha="center",
        va="bottom"
    )

plt.show()

In [ ]:
reward = comparison["Reward"]

plt.figure(figsize=(8, 5))

bars = plt.bar(models, reward)

plt.title("Comparação do reward total")
plt.ylabel("Reward")

for bar, value in zip(bars, reward):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(value),
        ha="center",
        va="bottom"
    )

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

labels = [
    ARMS[arm]
    for arm in arm_counts.index
]

plt.bar(
    labels,
    arm_counts.values
)

plt.title("Distribuição das ofertas escolhidas")
plt.xlabel("Oferta")
plt.ylabel("Quantidade de escolhas")

plt.show()

In [ ]:
def evaluate_experiment(test_df, probabilities, seed):
    np.random.seed(seed)

    baseline_rewards = run_baseline(
        test_df,
        probabilities
    )

    ts_rewards, selected_arms, bandit = run_thompson_sampling(
        test_df,
        probabilities
    )

    return {
        "seed": seed,
        "baseline_conversion": baseline_rewards.mean(),
        "ts_conversion": ts_rewards.mean(),
        "baseline_reward": baseline_rewards.sum(),
        "ts_reward": ts_rewards.sum()
    }

In [ ]:
experiments = []

for seed in range(30):
    result = evaluate_experiment(
        test_df,
        probabilities,
        seed
    )

    experiments.append(result)

experiments_df = pd.DataFrame(experiments)

experiments_df.head()

In [ ]:
experiments_df[
    [
        "baseline_conversion",
        "ts_conversion",
        "baseline_reward",
        "ts_reward"
    ]
].mean()

In [ ]:
experiments_df[
    [
        "baseline_conversion",
        "ts_conversion",
        "baseline_reward",
        "ts_reward"
    ]
].std()

In [ ]:
mean_baseline = experiments_df["baseline_conversion"].mean()
mean_ts = experiments_df["ts_conversion"].mean()

improvement = (
    (mean_ts - mean_baseline)
    / mean_baseline
) * 100

print(f"Baseline médio: {mean_baseline:.2%}")
print(f"Thompson Sampling médio: {mean_ts:.2%}")
print(f"Melhoria média: {improvement:.2f}%")

In [ ]:
arm_counts = pd.Series(selected_arms).value_counts().sort_index()

for arm, count in arm_counts.items():
    print(
        f"{ARMS[arm]}: "
        f"{count} "
        f"({count / len(selected_arms):.2%})"
    )

In [ ]:
print("Alpha:", bandit.alpha)
print("Beta:", bandit.beta)

### 4 - Test cases

In [ ]:
golden_set = test_df.sample(
    n=5,
    random_state=42
).copy()

golden_set

In [ ]:
golden_results = []

for idx, (_, row) in enumerate(golden_set.iterrows()):
    # Probabilidade base prevista pelo modelo
    base_probability = model.predict_proba(
        pd.DataFrame([row])[features]
    )[0, 1]

    # Probabilidade simulada para cada oferta
    arm_probabilities = {
        ARMS[arm]: get_arm_probability(
            row,
            base_probability,
            arm
        )
        for arm in ARMS
    }

    # Oferta com maior probabilidade simulada
    recommended_arm = max(
        arm_probabilities,
        key=arm_probabilities.get
    )

    golden_results.append({
        "cliente": f"Cliente {idx + 1}",
        "probabilidade_base": base_probability,
        "Oferta A": arm_probabilities["Oferta A"],
        "Oferta B": arm_probabilities["Oferta B"],
        "Oferta C": arm_probabilities["Oferta C"],
        "oferta_recomendada": recommended_arm
    })

golden_results_df = pd.DataFrame(golden_results)

golden_results_df

In [ ]:
golden_display = golden_results_df.copy()

probability_columns = [
    "probabilidade_base",
    "Oferta A",
    "Oferta B",
    "Oferta C"
]

for column in probability_columns:
    golden_display[column] = (
        golden_display[column] * 100
    ).round(2).astype(str) + "%"

golden_display

In [ ]:
def get_recommendation_reason(row, recommended_arm):
    reasons = []

    if recommended_arm == "Oferta B":
        reasons.append("abordagem personalizada")

        if row["education"] == "tertiary":
            reasons.append("educação terciária")

        if row["housing"] == "no":
            reasons.append("sem financiamento habitacional")

    elif recommended_arm == "Oferta C":
        reasons.append("abordagem alternativa")

        if row["poutcome"] == "success":
            reasons.append("resultado anterior positivo")

        if row["previous"] > 0:
            reasons.append("interações anteriores")

    else:
        reasons.append("estratégia padrão")

    return ", ".join(reasons)

In [ ]:
golden_display["justificativa"] = [
    get_recommendation_reason(
        golden_set.iloc[i],
        golden_results_df.iloc[i]["oferta_recomendada"]
    )
    for i in range(len(golden_set))
]

golden_display

In [ ]:
golden_candidates = test_df.copy()

golden_candidates["probabilidade_base"] = model.predict_proba(
    golden_candidates[features]
)[:, 1]

for arm, name in ARMS.items():
    golden_candidates[name] = golden_candidates.apply(
        lambda row: get_arm_probability(
            row,
            row["probabilidade_base"],
            arm
        ),
        axis=1
    )

golden_candidates["oferta_recomendada"] = golden_candidates[
    ["Oferta A", "Oferta B", "Oferta C"]
].idxmax(axis=1)

golden_candidates["oferta_recomendada"].value_counts()

In [ ]:
golden_set = pd.concat([
    golden_candidates[
        golden_candidates["oferta_recomendada"] == "Oferta A"
    ].head(2),

    golden_candidates[
        golden_candidates["oferta_recomendada"] == "Oferta B"
    ].head(2),

    golden_candidates[
        golden_candidates["oferta_recomendada"] == "Oferta C"
    ].head(2)
]).head(5)

golden_set

In [ ]:
def get_recommendation_reason(row):
    arm = row["oferta_recomendada"]
    reasons = []

    if arm == "Oferta A":
        return "estratégia padrão"

    if arm == "Oferta B":
        reasons.append("abordagem personalizada")

        if row["education"] == "tertiary":
            reasons.append("educação terciária")

        if row["housing"] == "no":
            reasons.append("sem financiamento habitacional")

    if arm == "Oferta C":
        reasons.append("abordagem alternativa")

        if row["poutcome"] == "success":
            reasons.append("resultado anterior positivo")

        if row["previous"] > 0:
            reasons.append("interações anteriores")

    return ", ".join(reasons)

In [ ]:
golden_display = golden_set[
    [
        "age",
        "job",
        "education",
        "housing",
        "previous",
        "poutcome",
        "probabilidade_base",
        "Oferta A",
        "Oferta B",
        "Oferta C",
        "oferta_recomendada"
    ]
].copy()

golden_display["justificativa"] = golden_set.apply(
    get_recommendation_reason,
    axis=1
)

probability_columns = [
    "probabilidade_base",
    "Oferta A",
    "Oferta B",
    "Oferta C"
]

for column in probability_columns:
    golden_display[column] = (
        golden_display[column] * 100
    ).round(2).astype(str) + "%"

golden_display

In [ ]:
golden_set = pd.concat([
    golden_candidates[
        golden_candidates["oferta_recomendada"] == "Oferta B"
    ].sample(n=3, random_state=42),

    golden_candidates[
        golden_candidates["oferta_recomendada"] == "Oferta C"
    ].sample(n=2, random_state=42)
]).reset_index(drop=True)

golden_set

In [ ]:
golden_display = golden_set[
    [
        "age",
        "job",
        "education",
        "housing",
        "previous",
        "poutcome",
        "probabilidade_base",
        "Oferta A",
        "Oferta B",
        "Oferta C",
        "oferta_recomendada"
    ]
].copy()

golden_display["justificativa"] = golden_set.apply(
    get_recommendation_reason,
    axis=1
)

probability_columns = [
    "probabilidade_base",
    "Oferta A",
    "Oferta B",
    "Oferta C"
]

for column in probability_columns:
    golden_display[column] = (
        golden_display[column] * 100
    ).round(2).astype(str) + "%"

golden_display